# Statistics Lab Manual
## Descriptive Statistics & Exploratory Data Analysis with Python (Jupyter Notebook)

**Dataset:** California Housing Dataset (real-world, ~20,640 records, U.S. Census 1990 block-group data)

**Topics covered:**
1. Problem Statement
2. Load Dataset
3. Dependent and Independent Variables
4. Mean vs Median comparison
5. Variance vs IQR
6. Skewness & Distribution Shape
7. Histogram, Boxplot, Density Plots
8. Pivot Tables

---


## 1. Problem Statement

A real-estate analytics firm wants to understand what drives **median house values** across
California neighborhoods (block groups) so that they can:

- Advise buyers/investors on fair pricing.
- Identify which socio-economic and geographic factors (income, house age, rooms,
  population, location) are associated with higher or lower housing prices.
- Understand the **distribution** of house prices (is it symmetric? skewed? are there
  outliers?) before building any predictive model, since many statistical/ML models
  assume roughly normal, low-skew data.

**Business question:** *"How is median house value distributed across California, and
which variables (median income, house age, rooms, population, location) are associated
with it?"*

This is a classic **regression-style EDA problem** — the target/dependent variable is
continuous (`median_house_value`), and we explore several independent variables that
might explain its variation, using descriptive statistics and visualizations.

## 2. Load Dataset

We use the **California Housing dataset**, a real dataset derived from the 1990 U.S. Census,
containing **20,640 rows** (one per census block group) and 8 numeric features plus the
target `median_house_value`.

It ships with `scikit-learn` (`sklearn.datasets.fetch_california_housing`) and is
downloaded/cached automatically the first time you run this notebook (internet required
once). A synthetic fallback generator (same schema, similar real-world statistical
properties) is included so the notebook still runs end-to-end if you are offline —
but for the real assignment, please run it with internet access at least once so the
real dataset gets cached locally.

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

print("Libraries loaded successfully.")

In [ ]:
def load_real_california_housing():
    """Try to load the REAL California Housing dataset from scikit-learn."""
    from sklearn.datasets import fetch_california_housing
    data = fetch_california_housing(as_frame=True)
    df_ = data.frame.copy()
    df_.rename(columns={"MedHouseVal": "median_house_value"}, inplace=True)
    df_["median_house_value"] = df_["median_house_value"] * 100000  # scale to USD
    # Add a categorical column for pivot-table demos
    def region(lat, lon):
        if lat > 37.5:
            return "Northern CA"
        elif lat > 35.0:
            return "Central CA"
        else:
            return "Southern CA"
    df_["region"] = [region(a, b) for a, b in zip(df_["Latitude"], df_["Longitude"])]
    return df_, "real"

def generate_synthetic_california_housing(n=20640, seed=42):
    """Offline fallback: synthetic dataset with realistic, skewed, real-world-like
    statistical structure matching the real dataset's schema (used ONLY if the
    real dataset cannot be downloaded, e.g. no internet access)."""
    rng = np.random.default_rng(seed)
    median_income = rng.gamma(shape=5.0, scale=0.8, size=n).clip(0.5, 15)          # right-skewed
    house_age = rng.uniform(1, 52, size=n)
    avg_rooms = rng.normal(5.4, 1.1, size=n).clip(1, 15)
    avg_bedrooms = avg_rooms * rng.uniform(0.15, 0.25, size=n)
    population = rng.lognormal(mean=6.9, sigma=0.6, size=n).clip(3, 15000)          # right-skewed
    avg_occup = rng.normal(3.0, 0.6, size=n).clip(1, 8)
    latitude = rng.uniform(32.5, 42.0, size=n)
    longitude = rng.uniform(-124.3, -114.3, size=n)

    noise = rng.normal(0, 0.5, size=n)
    median_house_value = (
        15000
        + median_income * 40000
        - house_age * 300
        + avg_rooms * 8000
        - avg_occup * 5000
        + noise * 20000
    ).clip(15000, 500001)

    def region(lat, lon):
        if lat > 37.5:
            return "Northern CA"
        elif lat > 35.0:
            return "Central CA"
        else:
            return "Southern CA"

    df_ = pd.DataFrame({
        "MedInc": median_income,
        "HouseAge": house_age,
        "AveRooms": avg_rooms,
        "AveBedrms": avg_bedrooms,
        "Population": population,
        "AveOccup": avg_occup,
        "Latitude": latitude,
        "Longitude": longitude,
        "median_house_value": median_house_value,
    })
    df_["region"] = [region(a, b) for a, b in zip(df_["Latitude"], df_["Longitude"])]
    return df_, "synthetic"

try:
    df, source_type = load_real_california_housing()
    print("Loaded the REAL California Housing dataset from scikit-learn.")
except Exception as e:
    print(f"Could not download the real dataset ({e}).")
    print("Falling back to a large SYNTHETIC dataset with realistic statistical structure.")
    df, source_type = generate_synthetic_california_housing()

print(f"Data source : {source_type}")
print(f"Shape       : {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

In [ ]:
# Basic structural overview
df.info()
print("\nMissing values per column:\n", df.isnull().sum())
df.describe().T

## 3. Dependent and Independent Variables

| Type | Variable | Meaning |
|---|---|---|
| **Dependent (target, Y)** | `median_house_value` | Median house value (USD) for the block group — what we want to explain/predict |
| **Independent (features, X)** | `MedInc` | Median household income in the block group (in tens of thousands USD) |
| | `HouseAge` | Median age of houses in the block group |
| | `AveRooms` | Average number of rooms per household |
| | `AveBedrms` | Average number of bedrooms per household |
| | `Population` | Block group population |
| | `AveOccup` | Average number of household members |
| | `Latitude`, `Longitude` | Geographic location |
| | `region` (derived) | Categorical: Northern / Central / Southern CA |

The **dependent variable** is the one whose behaviour we are trying to understand or
predict (`median_house_value`). The **independent variables** are the predictors/features
we believe influence it.

In [ ]:
dependent_variable = "median_house_value"
independent_variables = [c for c in df.columns if c not in [dependent_variable, "region"]]

y = df[dependent_variable]
X = df[independent_variables]

print("Dependent variable  (Y):", dependent_variable)
print("Independent variables (X):", independent_variables)

## 4. Mean vs Median Comparison

- **Mean** = arithmetic average — sensitive to outliers/extreme values.
- **Median** = middle value when sorted — robust to outliers.
- If **mean ≈ median** → distribution is roughly symmetric.
- If **mean > median** → distribution is right-skewed (a few very high values pull the mean up).
- If **mean < median** → distribution is left-skewed.

We compute both for the target and key independent variables to detect skew before doing
any modeling.

In [ ]:
cols_to_check = ["median_house_value", "MedInc", "Population", "AveRooms", "HouseAge"]

summary = pd.DataFrame({
    "mean": df[cols_to_check].mean(),
    "median": df[cols_to_check].median(),
})
summary["mean_minus_median"] = summary["mean"] - summary["median"]
summary["pct_difference"] = (summary["mean_minus_median"] / summary["median"] * 100).round(2)
summary

In [ ]:
for col in cols_to_check:
    diff = summary.loc[col, "mean_minus_median"]
    if abs(diff) < 0.02 * summary.loc[col, "median"]:
        verdict = "roughly SYMMETRIC (mean ~ median)"
    elif diff > 0:
        verdict = "RIGHT-SKEWED (mean > median, high-value outliers pull mean up)"
    else:
        verdict = "LEFT-SKEWED (mean < median, low-value outliers pull mean down)"
    print(f"{col:20s} -> mean={summary.loc[col,'mean']:,.2f}, "
          f"median={summary.loc[col,'median']:,.2f}  =>  {verdict}")

**Interpretation:** `median_house_value`, `MedInc`, and `Population` typically show
**mean > median**, i.e. right-skewed — a relatively small number of expensive,
high-income or densely populated block groups pull the mean upward. This means the
**median** is the more representative "typical value" for these variables, while the
mean is inflated by outliers on the high end.

## 5. Variance vs IQR (Interquartile Range)

- **Variance / Standard deviation**: average squared/absolute deviation from the mean.
  Uses *every* data point, so it is heavily influenced by outliers.
- **IQR = Q3 − Q1** (the spread of the middle 50% of the data): robust to outliers,
  a better measure of spread for skewed data.

A large gap between what variance/std "suggest" and what IQR "suggests" is itself a
signal of outliers or skew.

In [ ]:
spread_stats = pd.DataFrame({
    "variance": df[cols_to_check].var(),
    "std_dev": df[cols_to_check].std(),
    "Q1": df[cols_to_check].quantile(0.25),
    "Q3": df[cols_to_check].quantile(0.75),
})
spread_stats["IQR"] = spread_stats["Q3"] - spread_stats["Q1"]
spread_stats["std_to_IQR_ratio"] = (spread_stats["std_dev"] / spread_stats["IQR"]).round(2)
spread_stats

In [ ]:
# Outlier detection using the 1.5 x IQR rule (Tukey's fences)
for col in cols_to_check:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = n_outliers / len(df) * 100
    print(f"{col:20s} -> outliers beyond 1.5*IQR fences: {n_outliers:5d} ({pct:5.2f}% of data)")

**Interpretation:** For skewed variables like `median_house_value` and `Population`,
the standard deviation looks large mainly because a handful of extreme block groups
pull it up — the IQR gives a tighter, more trustworthy sense of "typical" spread for
the bulk of the data. A high **std/IQR ratio** flags variables where outliers dominate
the variance.

## 6. Skewness & Distribution Shape

**Skewness** quantifies asymmetry of a distribution:

| Skewness value | Interpretation |
|---|---|
| ≈ 0 | Symmetric (normal-like) |
| > 0 | Right (positive) skew — long tail toward high values |
| < 0 | Left (negative) skew — long tail toward low values |
| \|skew\| > 1 | Highly skewed |
| 0.5 < \|skew\| ≤ 1 | Moderately skewed |
| \|skew\| ≤ 0.5 | Approximately symmetric |

We also compute **kurtosis** (tail heaviness / peakedness relative to a normal
distribution) for additional context.

In [ ]:
shape_stats = pd.DataFrame({
    "skewness": df[cols_to_check].apply(lambda s: stats.skew(s.dropna())),
    "kurtosis": df[cols_to_check].apply(lambda s: stats.kurtosis(s.dropna())),
})

def skew_label(s):
    if abs(s) <= 0.5:
        return "approximately symmetric"
    elif abs(s) <= 1:
        return "moderately skewed"
    else:
        return "highly skewed"

shape_stats["shape"] = shape_stats["skewness"].apply(
    lambda s: f"{'right' if s > 0 else 'left'}-skewed, {skew_label(s)}" if abs(s) > 0.5 else "approximately symmetric"
)
shape_stats

**Interpretation:** `median_house_value`, `MedInc`, `Population`, and `AveRooms` are
typically **right-skewed** (positive skewness), confirming what the mean-vs-median
comparison already suggested. Positive kurtosis indicates heavier tails / more extreme
values than a normal distribution — reinforcing the need to check for outliers and
consider log-transformation before applying models that assume normality.

## 7. Histogram, Boxplot, and Density Plots

Visualizing distribution shape complements the numeric statistics above:

- **Histogram** — frequency of values in bins; shows overall shape, modality, skew.
- **Boxplot** — shows median, Q1/Q3, whiskers, and outliers at a glance.
- **Density (KDE) plot** — smoothed continuous version of the histogram, easier to
  compare shapes across variables/groups.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df["median_house_value"], bins=50, color="steelblue", edgecolor="white")
axes[0].axvline(df["median_house_value"].mean(), color="red", linestyle="--", label="Mean")
axes[0].axvline(df["median_house_value"].median(), color="green", linestyle="--", label="Median")
axes[0].set_title("Histogram: Median House Value")
axes[0].set_xlabel("Median House Value (USD)")
axes[0].legend()

sns.boxplot(y=df["median_house_value"], ax=axes[1], color="lightcoral")
axes[1].set_title("Boxplot: Median House Value")

sns.kdeplot(df["median_house_value"], ax=axes[2], fill=True, color="seagreen")
axes[2].set_title("Density Plot: Median House Value")
axes[2].set_xlabel("Median House Value (USD)")

plt.tight_layout()
plt.show()

In [ ]:
# Same trio of plots for a grid of independent variables
fig, axes = plt.subplots(len(cols_to_check[1:]), 3, figsize=(16, 4 * len(cols_to_check[1:])))

for i, col in enumerate(cols_to_check[1:]):
    axes[i, 0].hist(df[col], bins=40, color="steelblue", edgecolor="white")
    axes[i, 0].set_title(f"Histogram: {col}")

    sns.boxplot(y=df[col], ax=axes[i, 1], color="lightcoral")
    axes[i, 1].set_title(f"Boxplot: {col}")

    sns.kdeplot(df[col], ax=axes[i, 2], fill=True, color="seagreen")
    axes[i, 2].set_title(f"Density: {col}")

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot of house value grouped by region (categorical comparison)
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="region", y="median_house_value", palette="Set2")
plt.title("Median House Value by Region")
plt.ylabel("Median House Value (USD)")
plt.show()

**Interpretation:** The histogram and density plot both show a long right tail for
`median_house_value` (many moderately priced homes, a smaller number of very expensive
ones), consistent with the positive skewness computed earlier. The boxplot shows several
points above the upper whisker — statistical outliers — and the region-wise boxplot shows
that location shifts both the median and the spread of prices.

## 8. Pivot Tables

Pivot tables let us summarize the dependent variable across combinations of categorical
groupings — useful for spotting patterns that single-variable statistics miss. We bucket
some continuous variables (income, house age) into categories to build meaningful pivots.

In [ ]:
# Create categorical bins for pivoting
df["income_level"] = pd.qcut(df["MedInc"], q=4, labels=["Low", "Medium", "High", "Very High"])
df["age_group"] = pd.cut(df["HouseAge"], bins=[0, 15, 30, 45, 60],
                          labels=["0-15 yrs", "16-30 yrs", "31-45 yrs", "46-60 yrs"])

# Pivot 1: average house value by region and income level
pivot1 = pd.pivot_table(
    df, values="median_house_value", index="region", columns="income_level",
    aggfunc="mean"
)
pivot1

In [ ]:
# Pivot 2: median house value & count, by age group and income level
pivot2 = pd.pivot_table(
    df, values="median_house_value", index="age_group", columns="income_level",
    aggfunc=["median", "count"]
)
pivot2

In [ ]:
# Pivot 3: multiple summary statistics at once, by region
pivot3 = pd.pivot_table(
    df, values="median_house_value", index="region",
    aggfunc=["mean", "median", "std", "count"]
)
pivot3

In [ ]:
# Heatmap visualization of Pivot 1
plt.figure(figsize=(8, 5))
sns.heatmap(pivot1, annot=True, fmt=",.0f", cmap="YlOrRd")
plt.title("Average Median House Value by Region and Income Level")
plt.show()

**Interpretation:** The pivot tables show that median house value rises consistently
from the "Low" to "Very High" income level within every region, and that "Northern CA"
and "Southern CA" (which contain the San Francisco Bay Area and Los Angeles/San Diego
metro areas respectively) command higher average prices than "Central CA" at every
income level — location and income interact rather than acting independently.

## Lab Summary & Exercises

**What we did:**
1. Framed a real-world problem (what explains California house prices).
2. Loaded a real, large (20,640-row) dataset.
3. Identified dependent (`median_house_value`) vs independent variables.
4. Compared mean vs median to detect skew.
5. Compared variance vs IQR to detect outlier-driven spread.
6. Quantified skewness/kurtosis to formally describe distribution shape.
7. Visualized distributions with histograms, boxplots, and density plots.
8. Built pivot tables to summarize the target across categorical groupings.

**Try it yourself:**
1. Repeat the mean-vs-median and skewness analysis for `AveBedrms` and `Longitude`.
2. Log-transform `median_house_value` (`np.log1p`) and recompute skewness — how much
   does it change?
3. Build a pivot table of **average `AveRooms`** by `region` and `age_group`.
4. Using the 1.5×IQR rule, remove outliers from `Population` and compare the new mean
   vs median to the original.
5. Create a violin plot (`sns.violinplot`) of `median_house_value` by `income_level` and
   compare it to the boxplot version — what extra information does it show?